In [ ]:
import json
from decimal import Decimal, getcontext
from collections import defaultdict

getcontext().prec = 50

In [ ]:
with open("../data/v2pools.json") as f:
    pools = json.load(f)

print("Total pools:", len(pools))
pools[0]

In [ ]:
for i in range(5):
    p = pools[i]
    print(
        p["id"],
        p["token0"]["id"],
        p["token1"]["id"],
        p["reserve0"],
        p["reserve1"]
    )

In [ ]:
usd_values = []

for p in pools:
    try:
        usd_values.append(float(p["reserveUSD"]))
    except:
        pass

print("Min:", min(usd_values))
print("Max:", max(usd_values))

In [ ]:
import numpy as np
np.percentile(usd_values, [0,10,25,50,75,90,99])

In [ ]:
tokens = set()

for p in pools:
    tokens.add(p["token0"]["id"])
    tokens.add(p["token1"]["id"])

print("Unique tokens:", len(tokens))

In [ ]:
filtered_pools = []

for p in pools:
    try:
        reserve0 = Decimal(p["reserve0"])
        reserve1 = Decimal(p["reserve1"])
        reserve_usd = Decimal(p["reserveUSD"])

        if reserve0 <= 0 or reserve1 <= 0:
            continue

        if reserve_usd < Decimal("100"):
            continue

        filtered_pools.append(p)

    except:
        continue

print("Filtered pools:", len(filtered_pools))

In [ ]:
graph = defaultdict(list)

for p in filtered_pools:

    token0 = p["token0"]["id"]
    token1 = p["token1"]["id"]

    reserve0 = Decimal(p["reserve0"])
    reserve1 = Decimal(p["reserve1"])

    pool_id = p["id"]

    graph[token0].append({
        "to": token1,
        "reserve_in": reserve0,
        "reserve_out": reserve1,
        "pool": pool_id
    })

    graph[token1].append({
        "to": token0,
        "reserve_in": reserve1,
        "reserve_out": reserve0,
        "pool": pool_id
    })

In [ ]:
print("Graph nodes:", len(graph))

In [ ]:
degrees = [len(v) for v in graph.values()]

print("Max edges from token:", max(degrees))
print("Average edges:", sum(degrees)/len(degrees))

In [ ]:
def get_amount_out(amount_in, reserve_in, reserve_out, fee=Decimal("0.003")):
    
    amount_in_with_fee = amount_in * (Decimal("1") - fee)
    
    return (amount_in_with_fee * reserve_out) / (reserve_in + amount_in_with_fee)

In [ ]:
get_amount_out(
    Decimal("1"),
    Decimal("100"),
    Decimal("100")
)

In [ ]:
def find_triangular_arbitrage(graph, start_amount=Decimal("1")):
    
    opportunities = []
    
    for token_a in graph:
        
        for edge_ab in graph[token_a]:
            token_b = edge_ab["to"]
            
            for edge_bc in graph.get(token_b, []):
                token_c = edge_bc["to"]
                
                # avoid immediate backtracking
                if token_c == token_a:
                    continue
                
                for edge_ca in graph.get(token_c, []):
                    
                    if edge_ca["to"] != token_a:
                        continue
                    
                    # simulate cycle
                    amount = start_amount
                    
                    amount = get_amount_out(
                        amount,
                        edge_ab["reserve_in"],
                        edge_ab["reserve_out"]
                    )
                    
                    amount = get_amount_out(
                        amount,
                        edge_bc["reserve_in"],
                        edge_bc["reserve_out"]
                    )
                    
                    amount = get_amount_out(
                        amount,
                        edge_ca["reserve_in"],
                        edge_ca["reserve_out"]
                    )
                    
                    profit = amount - start_amount
                    
                    if profit > 0:
                        opportunities.append({
                            "cycle": [token_a, token_b, token_c, token_a],
                            "profit": profit,
                            "final_amount": amount
                        })
    
    return opportunities

In [ ]:
opps = find_triangular_arbitrage(graph)

print("Total profitable cycles:", len(opps))

In [ ]:
opps_sorted = sorted(
    opps,
    key=lambda x: x["profit"],
    reverse=True
)

for o in opps_sorted[:10]:
    print(o)

In [ ]:
def canonicalize_cycle_nodes(cycle):
    """
    Convert [A, B, C, A] into a rotation-invariant tuple like (A, B, C).
    Cycles that are the same up to rotation will map to the same key.
    """
    core = cycle[:-1]  # remove repeated last node
    rotations = [tuple(core[i:] + core[:i]) for i in range(len(core))]
    return min(rotations)

In [ ]:
def simulate_cycle_edges(edges, amount_in):
    """
    Simulate a full cycle using a fixed list of directed edges.
    Returns final amount after all swaps.
    """
    amount = amount_in
    for edge in edges:
        amount = get_amount_out(
            amount,
            edge["reserve_in"],
            edge["reserve_out"]
        )
    return amount

In [ ]:
def get_candidate_trade_sizes(edges):
    """
    Build a small set of sensible candidate input sizes based on the smallest
    input-side reserve across the cycle.

    We cap trade sizes to tiny fractions of liquidity to reduce nonsense profits.
    """
    min_reserve_in = min(edge["reserve_in"] for edge in edges)

    fractions = [
        Decimal("0.0001"),   # 0.01%
        Decimal("0.0005"),   # 0.05%
        Decimal("0.001"),    # 0.1%
        Decimal("0.005"),    # 0.5%
        Decimal("0.01"),     # 1%
    ]

    candidates = []
    for frac in fractions:
        amt = min_reserve_in * frac
        if amt > 0:
            candidates.append(amt)

    # deduplicate after quantization-ish via string form
    unique = []
    seen = set()
    for amt in candidates:
        key = str(amt.normalize())
        if key not in seen:
            seen.add(key)
            unique.append(amt)

    return unique

In [ ]:
def evaluate_cycle_with_best_trade_size(edges, cycle_nodes):
    """
    For one cycle, try several candidate trade sizes and keep the best result.
    """
    best = None

    candidate_sizes = get_candidate_trade_sizes(edges)

    for amount_in in candidate_sizes:
        final_amount = simulate_cycle_edges(edges, amount_in)
        profit = final_amount - amount_in

        if profit > 0:
            profit_pct = (profit / amount_in) * Decimal("100")

            result = {
                "cycle": cycle_nodes,
                "amount_in": amount_in,
                "final_amount": final_amount,
                "profit": profit,
                "profit_pct": profit_pct,
                "edges": edges,
            }

            if best is None or result["profit"] > best["profit"]:
                best = result

    return best

In [ ]:
def find_triangular_arbitrage_deduped(graph):
    """
    Find triangular arbitrage cycles A -> B -> C -> A,
    deduplicate equivalent rotations,
    and evaluate each cycle with a small trade-size search.
    """
    opportunities = []
    seen_cycles = set()

    for token_a in graph:
        for edge_ab in graph[token_a]:
            token_b = edge_ab["to"]

            for edge_bc in graph.get(token_b, []):
                token_c = edge_bc["to"]

                # Skip immediate backtracking
                if token_c == token_a:
                    continue

                for edge_ca in graph.get(token_c, []):
                    if edge_ca["to"] != token_a:
                        continue

                    cycle_nodes = [token_a, token_b, token_c, token_a]
                    cycle_key = canonicalize_cycle_nodes(cycle_nodes)

                    if cycle_key in seen_cycles:
                        continue

                    seen_cycles.add(cycle_key)

                    edges = [edge_ab, edge_bc, edge_ca]
                    best_result = evaluate_cycle_with_best_trade_size(edges, cycle_nodes)

                    if best_result is not None:
                        opportunities.append(best_result)

    return opportunities

In [ ]:
opps_deduped = find_triangular_arbitrage_deduped(graph)

print("Deduped profitable cycles:", len(opps_deduped))

In [ ]:
opps_deduped_sorted = sorted(
    opps_deduped,
    key=lambda x: x["profit"],
    reverse=True
)

for o in opps_deduped_sorted[:10]:
    print("Cycle:", " -> ".join(o["cycle"]))
    print("Amount in:", o["amount_in"])
    print("Final amount:", o["final_amount"])
    print("Profit:", o["profit"])
    print("Profit %:", o["profit_pct"])
    print("-" * 80)

In [ ]:
def short_token(token):
    return f"{token[:6]}...{token[-4:]}"

In [ ]:
top10 = sorted(opps_deduped, key=lambda x: x["profit"], reverse=True)[:10]

for i, o in enumerate(top10, start=1):
    pretty_cycle = " -> ".join(short_token(t) for t in o["cycle"])
    print(f"#{i}")
    print("Cycle:      ", pretty_cycle)
    print("Amount in:  ", o["amount_in"])
    print("Final amt:  ", o["final_amount"])
    print("Profit:     ", o["profit"])
    print("Profit %:   ", o["profit_pct"])
    print("-" * 80)